# অলীকবচন v12 — LoRA-Stacked Judge (Team Outliers)

v11 (0.801 LB; ensemble 0.81) + a **QLoRA-trained verifier** as a third scoring component.

Per non-math row, three signals: **LoRA-14B Yes/No logprob** (task-specialised), **32B Yes/No
logprob** (world knowledge), **32B CoT self-consistency** (reasoning, with RAG evidence on
closed-book). Per-route blend weights and thresholds are fitted on the labelled sample split
by 5-fold CV. Stage order: the LoRA pass runs FIRST on transformers+bnb (then is freed), so
vLLM never has to be torn down.

**Attach:** competition data · LoRA adapter dataset (from `train_lora_v12.ipynb`) ·
wiki/GK corpora (multiple allowed — all are merged) · (optional) vLLM wheels.
Missing adapter → v12 degrades gracefully to v11 behaviour.

Compliance: open weights, local inference, T4×2 <9h, <50GB, no test-set training, external data
public+declared. This single notebook reproduces the selected submission end-to-end (Phase-2 rule).


In [ ]:
# ---- engine install: MUST run before torch is imported anywhere ----
import importlib.util, subprocess, glob as _g, os
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
if importlib.util.find_spec("vllm") is None:
    whls = sorted(_g.glob("/kaggle/input/**/vllm*.whl", recursive=True))
    if whls:
        subprocess.run(["pip", "install", "-q", "--no-index", "--find-links",
                        whls[0].rsplit("/", 1)[0], "vllm"], check=False)
    else:
        subprocess.run(["pip", "install", "-q", "vllm"], check=False)
for pkg in ("bitsandbytes", "peft"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run(["pip", "install", "-q", "-U", pkg, "accelerate"], check=False)
print("engine install cell done")


In [ ]:
import os, re, gc, json, math, time, glob, hashlib, random
import numpy as np, pandas as pd

SEED = 42
random.seed(SEED); np.random.seed(SEED)
METRIC        = "macro"      # flip to "f1_0" if the all-zeros diagnostic says class-0 F1
K_GROUNDED, K_CLOSEDBOOK, K_MATH = 3, 7, 3
COT_TOKENS, MATH_TOKENS = 260, 560
CTX_CLIP, Q_CLIP, R_CLIP = 2000, 500, 700
EVIDENCE_K, EVIDENCE_CHARS = 4, 450
JUDGE_CHUNK, RAG_MAX_PASSAGES = 256, 500_000
JUDGE_LADDER = [("Qwen/Qwen2.5-32B-Instruct-AWQ", 2, 3072),
                ("Qwen/Qwen2.5-32B-Instruct-AWQ", 2, 2560),
                ("Qwen/Qwen2.5-14B-Instruct-AWQ", 2, 4096)]
LORA_BASE = "Qwen/Qwen2.5-14B-Instruct"

os.environ.setdefault("PYTORCH_NVFUSER_DISABLE", "fallback")
try:
    import torch
    torch.manual_seed(SEED)
    for _fn, _a in [("_jit_set_texpr_fuser_enabled", False), ("_jit_override_can_fuse_on_gpu", False),
                    ("_jit_override_can_fuse_on_cpu", False), ("_jit_set_nvfuser_enabled", False)]:
        try: getattr(torch._C, _fn)(_a)
        except Exception: pass
    N_GPU = max(torch.cuda.device_count(), 1)
except Exception:
    torch, N_GPU = None, 0

def find_test_csv():
    c = [p for p in sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True))
         if os.path.basename(p).lower().replace(" ", "").startswith("test")]
    assert c, "test csv not found"
    return c[0]

def find_labeled_sample():
    for p in sorted(glob.glob("/kaggle/input/**/*", recursive=True)):
        b = os.path.basename(p).lower()
        if b.replace(" ", "").startswith("train") and b.endswith(".csv"): return p
    for p in sorted(glob.glob("/kaggle/input/**/*.json", recursive=True)):
        b = os.path.basename(p).lower()
        if "sample" in b and "submission" not in b and "adapter" not in b: return p
    return None

def find_lora():
    for p in sorted(glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)):
        return os.path.dirname(p)
    return None

def find_wikis():
    hits = sorted(glob.glob("/kaggle/input/**/*passages*.parquet", recursive=True))
    bad = ("testset", "test set", "datasetsamples", "dataset samples", "samplesubmission",
           "sample submission", "synthetic", "submission", "adapter")
    cand = list(hits)
    for ext in ("*.parquet", "*.csv", "*.json", "*.jsonl", "*.txt", "*.tsv"):
        for p in glob.glob(f"/kaggle/input/**/{ext}", recursive=True):
            pl = p.lower()
            if any(b in pl for b in bad) or p in cand: continue
            if any(k in pl for k in ("wiki", "passage", "corpus", "bangla", "bengali", "article", "gk"))                     and os.path.getsize(p) > 1_000_000:
                cand.append(p)
    return sorted(set(cand), key=os.path.getsize, reverse=True)[:6]

TEST_CSV, SAMPLE_LB = find_test_csv(), find_labeled_sample()
LORA_DIR, WIKI_SRCS = find_lora(), find_wikis()
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
print("test:", TEST_CSV, "\nlabeled:", SAMPLE_LB, "\nlora:", LORA_DIR or "NONE (v11 behaviour)",
      "\nwikis:", WIKI_SRCS or "NONE", "\nGPUs:", N_GPU)


In [ ]:
BN2EN = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
MATH_KW = re.compile(
    r"শতকরা|সম্ভাবনা|গুণিতক|মৌলিক|ধারাটি|সমষ্টি|যোগফল|বর্গ(?:মূল|ইঞ্চি|ফুট)?|"
    r"ক্ষতি হ|লাভ হ|বয়স|কত ?দিনে|গড়|অনুপাত|ভগ্নাংশ|সুদ|আসল|"
    r"[0-9০-৯]\s*[+\-*/=]|√|x\s*[*+=]|\*\*|%")

def has_context(v):
    if v is None or (isinstance(v, float) and np.isnan(v)): return False
    return str(v).strip() not in ("", "[NULL]", "NULL", "nan")

def route_row(ctx, q):
    if MATH_KW.search(str(q)): return "math"
    return "grounded" if has_context(ctx) else "closedbook"

def _clip(s, n):
    s = str(s) if s is not None else ""
    return s if len(s) <= n else s[:n] + "…"

SYS = ("You are a meticulous bilingual (Bengali/English) fact-checker. You judge whether a "
       "Bengali answer is factually correct and, when a passage is given, supported by it.")
TRAP_RULES = (
    "Rules:\n"
    "- The response is faithful ONLY if it answers exactly what the question asks AND is factually correct.\n"
    "- Copying words from the passage does NOT make it faithful: if the question asks for a year, a name "
    "is wrong; if the question's premise is not supported by the passage, be skeptical.\n"
    "- Partially wrong, wrong entity/date/number, or fabricated details => hallucinated.\n")

def lp_prompt(ctx, q, r):
    """Single-token Yes/No verification prompt (evidence-free; same for train + LoRA inference)."""
    if has_context(ctx):
        return (f"Passage (Bengali):\n{_clip(ctx, 1400)}\n\nQuestion (Bengali): {_clip(q, 500)}\n"
                f"Response (Bengali): {_clip(r, 700)}\n\n" + TRAP_RULES +
                "Is the response faithful? Answer with exactly one word: Yes or No.")
    return ("Judge using your own knowledge of Bengali/Bangladesh facts.\n"
            f"Question (Bengali): {_clip(q, 500)}\nProposed answer (Bengali): {_clip(r, 700)}\n"
            "Is the proposed answer factually correct? Answer with exactly one word: Yes or No.\nVerdict:")

import numpy as _np

def load_frame(path):
    df = pd.read_json(path) if path.endswith(".json") else pd.read_csv(path)
    low = {c.lower().strip(): c for c in df.columns}
    cols = dict(ctx=low.get("context"), q=low.get("prompt_bn") or low.get("prompt"),
                r=low.get("response_bn") or low.get("response"),
                id=low.get("id"), y=low.get("label"))
    assert cols["q"] and cols["r"], f"unexpected schema: {list(df.columns)}"
    rts = np.array([route_row(df[cols["ctx"]].iloc[i] if cols["ctx"] else None,
                              df[cols["q"]].iloc[i]) for i in range(len(df))])
    return df, cols, rts

test, tcols, test_routes = load_frame(TEST_CSV)
ids = test[tcols["id"]].values if tcols["id"] else np.arange(1, len(test) + 1)
print(len(test), "test rows | routes:", pd.Series(test_routes).value_counts().to_dict())
train, rcols, train_routes = None, None, None
if SAMPLE_LB:
    train, rcols, train_routes = load_frame(SAMPLE_LB)
    if rcols["y"] is None: train = None
    else: print(len(train), "labeled rows")


In [ ]:
# ---------------- BM25 over ALL attached corpora (merged) ----------------
TOK_RE = re.compile(r"[\wঀ-৿]+")
STOP = set("কী কি কে কার কাকে কোন কোথায় কবে হয় ছিল ছিলেন এর একটি ও এবং থেকে সালে করে করা হয়েছিল যে এই".split())

def tokenize(s):
    return [t for t in TOK_RE.findall(str(s).translate(BN2EN).lower()) if t not in STOP and len(t) > 1]

class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.docs, self.k1, self.b = docs, k1, b
        self.post, self.dl = {}, np.zeros(len(docs), dtype=np.float32)
        for i, d in enumerate(docs):
            toks = tokenize(d); self.dl[i] = len(toks) or 1
            tf = {}
            for t in toks: tf[t] = tf.get(t, 0) + 1
            for t, f in tf.items(): self.post.setdefault(t, []).append((i, f))
        self.avgdl = float(self.dl.mean()) if len(docs) else 1.0
        self.N = len(docs)
    def top(self, query, k=4):
        sc = {}
        for t in set(tokenize(query)):
            pl = self.post.get(t)
            if not pl: continue
            idf = np.log(1 + (self.N - len(pl) + 0.5) / (len(pl) + 0.5))
            for i, f in pl:
                den = f + self.k1 * (1 - self.b + self.b * self.dl[i] / self.avgdl)
                sc[i] = sc.get(i, 0.0) + idf * f * (self.k1 + 1) / den
        return [self.docs[i] for i, _ in sorted(sc.items(), key=lambda x: -x[1])[:k]]

def load_corpus(path, out):
    def add(text, title=""):
        text = str(text)
        for s0 in range(0, min(len(text), 1800), 900):
            ch = text[s0:s0 + 900].strip()
            if len(ch) >= 80: out.append((title + ": " if title else "") + ch)
    try:
        if path.endswith(".txt"):
            for block in open(path, encoding="utf-8", errors="ignore").read().split("\n\n"):
                add(block)
                if len(out) >= RAG_MAX_PASSAGES: return
            return
        if path.endswith(".parquet"): d = pd.read_parquet(path)
        elif path.endswith(".tsv"):   d = pd.read_csv(path, sep="\t", on_bad_lines="skip")
        elif path.endswith(".csv"):   d = pd.read_csv(path, on_bad_lines="skip")
        else:
            try: d = pd.read_json(path, lines=path.endswith(".jsonl"))
            except ValueError: d = pd.read_json(path, lines=True)
        obj = [c for c in d.columns if d[c].dtype == object]
        tcol = max(obj, key=lambda c: d[c].astype(str).str.len().mean()) if obj else d.columns[-1]
        ttl = next((c for c in obj if c != tcol and "title" in str(c).lower()), None)
        for _, row in d.iterrows():
            add(row[tcol], str(row[ttl]) if ttl else "")
            if len(out) >= RAG_MAX_PASSAGES: return
    except Exception as e:
        print(f"  (skipping {os.path.basename(path)}: {str(e)[:90]})")

bm25 = None
if WIKI_SRCS:
    t0, _p = time.time(), []
    for w in WIKI_SRCS:
        n0 = len(_p); load_corpus(w, _p)
        print(f"  {os.path.basename(w)}: +{len(_p)-n0} passages")
    bm25 = BM25(_p); gc.collect()
    print(f"BM25 over {len(_p)} passages in {time.time()-t0:.0f}s")


In [ ]:
# ---------------- STAGE B FIRST: LoRA verifier logprob pass (transformers+bnb, then freed) ----------------
lora_test = np.full(len(test), np.nan, dtype=np.float32)
lora_train = np.full(len(train), np.nan, dtype=np.float32) if train is not None else None

if LORA_DIR and torch is not None:
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import PeftModel
    ltok = AutoTokenizer.from_pretrained(LORA_DIR)
    if ltok.pad_token is None: ltok.pad_token = ltok.eos_token
    ltok.padding_side = "left"
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                             bnb_4bit_quant_type="nf4")
    lbase = AutoModelForCausalLM.from_pretrained(LORA_BASE, device_map="auto",
                                                 quantization_config=bnb, low_cpu_mem_usage=True)
    lmod = PeftModel.from_pretrained(lbase, LORA_DIR); lmod.eval()
    yes_id = ltok.encode(" Yes", add_special_tokens=False)[0]
    no_id  = ltok.encode(" No",  add_special_tokens=False)[0]

    def lora_scores(df, cols, rts):
        get = lambda c, i: df[cols[c]].iloc[i] if cols[c] else None
        msgs = [ltok.apply_chat_template(
                    [{"role": "system", "content": SYS},
                     {"role": "user", "content": lp_prompt(get("ctx", i), get("q", i), get("r", i))}],
                    tokenize=False, add_generation_prompt=True) for i in range(len(df))]
        out = np.zeros(len(df), dtype=np.float32)
        for s0 in range(0, len(msgs), 8):
            enc = ltok(msgs[s0:s0+8], return_tensors="pt", padding=True,
                       truncation=True, max_length=1400).to(lmod.device)
            with torch.no_grad():
                lg = lmod(**enc).logits
            last = enc["attention_mask"].sum(1) - 1
            for bi in range(lg.shape[0]):
                pr = torch.softmax(lg[bi, last[bi], [yes_id, no_id]].float(), -1)
                out[s0 + bi] = float(pr[0])
            if s0 % 400 == 0: print(f"    lora {s0}/{len(msgs)}", flush=True)
        return out

    t0 = time.time()
    lora_test = lora_scores(test, tcols, test_routes)
    if train is not None:
        lora_train = lora_scores(train, rcols, train_routes)
    print(f"LoRA pass done in {(time.time()-t0)/60:.1f} min")
    del lmod, lbase; gc.collect(); torch.cuda.empty_cache()
else:
    print("no LoRA adapter attached -> v11 behaviour (lora component skipped)")
HAS_LORA = LORA_DIR is not None and not np.isnan(lora_test).all()


In [ ]:
# ---------------- STAGE A: vLLM 32B judge (loaded after the LoRA pass freed its memory) ----------------
from transformers import AutoTokenizer
ENGINE, LLM_OBJ, MODEL_SRC, MODEL_MAXLEN, tok = None, None, None, 3072, None

def _try_vllm():
    global ENGINE, LLM_OBJ, MODEL_SRC, MODEL_MAXLEN
    try:
        import vllm  # noqa
    except ImportError:
        return False
    from vllm import LLM
    for src, tp, ml in JUDGE_LADDER:
        if tp > max(N_GPU, 1): continue
        try:
            print(f">>> trying {src} (tp={tp}, max_len={ml})")
            LLM_OBJ = LLM(model=src, quantization="awq" if "awq" in src.lower() else None,
                          dtype="half", tensor_parallel_size=tp, max_model_len=ml,
                          gpu_memory_utilization=0.90, enforce_eager=True,
                          swap_space=2, disable_log_stats=True)
            ENGINE, MODEL_SRC, MODEL_MAXLEN = "vllm", src, ml
            return True
        except Exception as e:
            print(f"    failed: {type(e).__name__}: {str(e)[:160]}")
            LLM_OBJ = None; gc.collect()
            if torch is not None: torch.cuda.empty_cache()
    return False

assert _try_vllm(), "vLLM engine failed on every ladder entry — check GPU/accelerator settings"
tok = AutoTokenizer.from_pretrained(MODEL_SRC)
if tok.pad_token is None: tok.pad_token = tok.eos_token
print("engine:", ENGINE, "| model:", MODEL_SRC, "| max_len:", MODEL_MAXLEN)

def chatfmt(u):
    return tok.apply_chat_template([{"role": "system", "content": SYS},
                                    {"role": "user", "content": u}],
                                   tokenize=False, add_generation_prompt=True)

def _fit(txt, out_tokens):
    ids = tok.encode(txt)
    lim = MODEL_MAXLEN - out_tokens - 8
    return tok.decode(ids[:lim]) if len(ids) > lim else txt

YES_SET, NO_SET = {"yes", "y", "true"}, {"no", "n", "false"}

def logprob_pfaithful(user_msgs):
    from vllm import SamplingParams
    prompts = [_fit(chatfmt(u), 4) for u in user_msgs]
    outs = LLM_OBJ.generate(prompts, SamplingParams(temperature=0.0, max_tokens=1, logprobs=20))
    res = []
    for o in outs:
        py = pn = 0.0
        lp = o.outputs[0].logprobs
        if lp:
            for tid, l in lp[0].items():
                s = (l.decoded_token if l.decoded_token is not None else tok.decode([tid])).strip().lower()
                if s in YES_SET: py += math.exp(l.logprob)
                elif s in NO_SET: pn += math.exp(l.logprob)
        res.append(py / (py + pn) if (py + pn) > 1e-9 else 0.5)
    return res

def sample_texts(user_msgs, max_tokens, k):
    from vllm import SamplingParams
    prompts = [_fit(chatfmt(u), max_tokens) for u in user_msgs]
    sp = SamplingParams(temperature=0.7, top_p=0.9, max_tokens=max_tokens, n=k)
    return [[c.text for c in o.outputs] for o in LLM_OBJ.generate(prompts, sp)]

CACHE_PATH = os.path.join(WORK, f"judge_cache_{(MODEL_SRC or 'x').replace('/','_')}.json")
try: _cache = json.load(open(CACHE_PATH))
except Exception: _cache = {}

def _chunked(fn, msgs, kind, **kw):
    keys = [kind + hashlib.sha1(m.encode()).hexdigest() for m in msgs]
    out = [_cache.get(k) for k in keys]
    pend = [i for i, v in enumerate(out) if v is None]
    for s0 in range(0, len(pend), JUDGE_CHUNK):
        blk = pend[s0:s0 + JUDGE_CHUNK]
        for i, r in zip(blk, fn([msgs[i] for i in blk], **kw)):
            out[i] = r; _cache[keys[i]] = r
        json.dump(_cache, open(CACHE_PATH, "w"))
        print(f"    {kind} {min(s0+JUDGE_CHUNK, len(pend))}/{len(pend)}", flush=True)
    return out


In [ ]:
# ---------------- 32B prompts (CoT + evidence variants) ----------------
FEWSHOT_IDX, FEWSHOT = [], ""
if train is not None:
    _nc = train[[not has_context(train[rcols["ctx"]].iloc[i]) if rcols["ctx"] else True
                 for i in range(len(train))]].copy()
    if len(_nc) >= 4:
        _nc["tl"] = _nc[rcols["q"]].astype(str).str.len() + _nc[rcols["r"]].astype(str).str.len()
        FEWSHOT_IDX = (list(_nc[_nc[rcols["y"]] == 1].sort_values("tl").index[:2]) +
                       list(_nc[_nc[rcols["y"]] == 0].sort_values("tl").index[:2]))
        FEWSHOT = "\n\n".join(
            f"Question: {_clip(train.at[i, rcols['q']], 200)}\n"
            f"Proposed answer: {_clip(train.at[i, rcols['r']], 200)}\n"
            f"Verdict: {'Yes' if int(train.at[i, rcols['y']]) == 1 else 'No'}" for i in FEWSHOT_IDX)
print("few-shot rows (excluded from calibration):", FEWSHOT_IDX)

def p_grounded(ctx, q, r, want):
    base = (f"Passage (Bengali):\n{_clip(ctx, CTX_CLIP)}\n\n"
            f"Question (Bengali): {_clip(q, Q_CLIP)}\nResponse (Bengali): {_clip(r, R_CLIP)}\n\n" + TRAP_RULES)
    if want == "lp":
        return base + "Is the response faithful? Answer with exactly one word: Yes or No."
    return base + ("Reason in 1-3 short sentences, then on a new line output exactly "
                   "\"Verdict: Yes\" (faithful) or \"Verdict: No\" (hallucinated).")

def p_closedbook(q, r, ev, want):
    head = ("Solved examples:\n\n" + FEWSHOT + "\n\n") if FEWSHOT else ""
    evb = ("Evidence passages (Bengali Wikipedia):\n" +
           "\n".join("- " + _clip(e, EVIDENCE_CHARS) for e in ev) + "\n\n") if ev else ""
    base = (head + evb + "Judge using your own knowledge of Bengali/Bangladesh facts"
            + (" and the evidence above" if ev else "") +
            f".\nQuestion (Bengali): {_clip(q, Q_CLIP)}\nProposed answer (Bengali): {_clip(r, R_CLIP)}\n")
    if want == "lp":
        return base + "Is the proposed answer factually correct? Answer with exactly one word: Yes or No.\nVerdict:"
    return base + ("Reason in 1-3 short sentences, then on a new line output exactly "
                   "\"Verdict: Yes\" (correct) or \"Verdict: No\" (wrong/fabricated).")

def p_math(q, r):
    return ("Solve this Bengali math problem step by step yourself. Then compare YOUR answer with the "
            "proposed answer (treat equal values written differently as a match, e.g. ১/২ = 0.5).\n"
            f"Problem (Bengali): {_clip(q, 900)}\nProposed answer (Bengali): {_clip(r, R_CLIP)}\n"
            "End with exactly \"Verdict: Yes\" if they match, or \"Verdict: No\" if they do not.")

VERDICT_EN = re.compile(r"verdict\s*[:：]?\s*(yes|no)", re.I)

def parse_verdict(text):
    m = VERDICT_EN.findall(str(text))
    if m: return 1 if m[-1].lower() == "yes" else 0
    tl = str(text).strip().lower()
    if tl.startswith("yes"): return 1
    if tl.startswith("no"): return 0
    return None

def votes_score(comps):
    vs = [v for v in (parse_verdict(c) for c in comps) if v is not None]
    return (float(np.mean(vs)) if vs else None)


In [ ]:
# ---------------- 32B component scoring (lp32 + cot32 per row) ----------------
def score_components(df, cols, rts, tag):
    n = len(df)
    get = lambda c, i: df[cols[c]].iloc[i] if cols[c] else None
    lp32 = np.full(n, np.nan, dtype=np.float32)
    cot32 = np.full(n, np.nan, dtype=np.float32)
    idx = {"grounded": [], "closedbook": [], "math": []}
    for i in range(n): idx[rts[i]].append(i)
    ev_cache = {}
    def evidence(i):
        if bm25 is None: return []
        if i not in ev_cache:
            ev_cache[i] = bm25.top(str(get("q", i)) + " " + str(get("r", i)), k=EVIDENCE_K)
        return ev_cache[i]
    g, c, m = idx["grounded"], idx["closedbook"], idx["math"]
    if g:
        for j, v in zip(g, _chunked(logprob_pfaithful,
                [p_grounded(get("ctx", i), get("q", i), get("r", i), "lp") for i in g], f"{tag}-g-lp")):
            lp32[j] = v
        for j, comps in zip(g, _chunked(sample_texts,
                [p_grounded(get("ctx", i), get("q", i), get("r", i), "cot") for i in g],
                f"{tag}-g-cot", max_tokens=COT_TOKENS, k=K_GROUNDED)):
            v = votes_score(comps); cot32[j] = v if v is not None else np.nan
    if c:
        for j, v in zip(c, _chunked(logprob_pfaithful,
                [p_closedbook(get("q", i), get("r", i), evidence(i), "lp") for i in c], f"{tag}-c-lp")):
            lp32[j] = v
        for j, comps in zip(c, _chunked(sample_texts,
                [p_closedbook(get("q", i), get("r", i), evidence(i), "cot") for i in c],
                f"{tag}-c-cot", max_tokens=COT_TOKENS, k=K_CLOSEDBOOK)):
            v = votes_score(comps); cot32[j] = v if v is not None else np.nan
    if m:
        for j, comps in zip(m, _chunked(sample_texts,
                [p_math(get("q", i), get("r", i)) for i in m], f"{tag}-math",
                max_tokens=MATH_TOKENS, k=K_MATH)):
            v = votes_score(comps)
            lp32[j] = cot32[j] = (v if v is not None else 0.5)
    return lp32, cot32

t0 = time.time()
lp32_te, cot32_te = score_components(test, tcols, test_routes, "test")
lp32_tr = cot32_tr = None
if train is not None:
    lp32_tr, cot32_tr = score_components(train, rcols, train_routes, "calib")
print(f"32B components done in {(time.time()-t0)/60:.1f} min")


In [ ]:
# ---------------- per-route blend-weight + threshold fitting (5-fold CV) ----------------
def f1_class(y, p, cls):
    y, p = np.asarray(y), np.asarray(p)
    tp = ((p == cls) & (y == cls)).sum(); fp = ((p == cls) & (y != cls)).sum()
    fn = ((p != cls) & (y == cls)).sum()
    return 2*tp / (2*tp + fp + fn) if (2*tp + fp + fn) else 0.0
f1_0 = lambda y, p: f1_class(y, p, 0)
f1_macro = lambda y, p: (f1_class(y, p, 0) + f1_class(y, p, 1)) / 2
metric_fn = f1_macro if METRIC == "macro" else f1_0

def stack(lp, cot, lo, w):
    """w = (w_lp, w_cot, w_lora); NaN components are renormalized away per row."""
    comp = np.stack([lp, cot, lo]); wt = np.array(w, dtype=np.float32)[:, None]
    mask = ~np.isnan(comp)
    wt = wt * mask
    s = (np.nan_to_num(comp) * wt).sum(0) / np.clip(wt.sum(0), 1e-6, None)
    return s

W_GRID = [(a/10, b/10, 1 - a/10 - b/10)
          for a in range(0, 11) for b in range(0, 11 - a)]

def best_wt(lp, cot, lo, y):
    best = ((1/3, 1/3, 1/3), 0.5, -1)
    for w in W_GRID:
        s = stack(lp, cot, lo, w)
        for t in np.linspace(0.05, 0.95, 46):
            v = metric_fn(y, (s >= t).astype(int))
            if v > best[2]: best = (w, float(t), v)
    return best

WEIGHTS = {rt: (0.5, 0.5, 0.0) for rt in ("grounded", "closedbook", "math")}
THRESHOLDS = {rt: 0.5 for rt in ("grounded", "closedbook", "math")}
if train is not None:
    y = train[rcols["y"]].values.astype(int)
    lo_tr = lora_train if HAS_LORA else np.full(len(train), np.nan, dtype=np.float32)
    keep = np.array([i not in set(FEWSHOT_IDX) for i in train.index])
    rng = np.random.RandomState(SEED); folds = rng.permutation(len(train)) % 5
    oof = np.zeros(len(train), dtype=int)
    for k in range(5):
        trm, vam = (folds != k) & keep, (folds == k) & keep
        for rt in WEIGHTS:
            mtr = trm & (train_routes == rt); mva = vam & (train_routes == rt)
            if mtr.sum() >= 8:
                w, t, _ = best_wt(lp32_tr[mtr], cot32_tr[mtr], lo_tr[mtr], y[mtr])
            else:
                w, t = (0.5, 0.5, 0.0), 0.5
            if mva.sum():
                oof[mva] = (stack(lp32_tr[mva], cot32_tr[mva], lo_tr[mva], w) >= t).astype(int)
    print(f"5-fold CV (honest): macro={f1_macro(y[keep], oof[keep]):.4f}  "
          f"F1(0)={f1_0(y[keep], oof[keep]):.4f}")
    for rt in WEIGHTS:
        m = keep & (train_routes == rt)
        if m.sum() >= 10:
            w, t, v = best_wt(lp32_tr[m], cot32_tr[m], lo_tr[m], y[m])
            WEIGHTS[rt], THRESHOLDS[rt] = w, t
            print(f"{rt:<10} n={m.sum():>3} w(lp32,cot32,lora)=({w[0]:.1f},{w[1]:.1f},{w[2]:.1f}) "
                  f"t={t:.2f} {METRIC}={v:.3f}")
    pd.DataFrame({"route": train_routes, "lp32": lp32_tr, "cot32": cot32_tr,
                  "lora": lo_tr, "label": y}).to_csv(os.path.join(WORK, "calib_scores.csv"), index=False)
else:
    print("no labeled sample -> defaults", WEIGHTS, THRESHOLDS)


In [ ]:
# ---------------- submission ----------------
lo_te = lora_test if HAS_LORA else np.full(len(test), np.nan, dtype=np.float32)
final = np.zeros(len(test), dtype=np.float32)
for rt in WEIGHTS:
    m = test_routes == rt
    final[m] = stack(lp32_te[m], cot32_te[m], lo_te[m], WEIGHTS[rt])
labels = np.array([int(final[i] >= THRESHOLDS[test_routes[i]]) for i in range(len(test))])
sub = pd.DataFrame({"id": ids, "label": labels})
assert list(sub.columns) == ["id", "label"] and len(sub) == len(test)
assert sub["label"].isin([0, 1]).all() and sub["id"].notna().all()
sub.to_csv(os.path.join(WORK, "submission.csv"), index=False)
pd.DataFrame({"id": ids, "route": test_routes, "lp32": lp32_te, "cot32": cot32_te,
              "lora": lo_te, "score": final, "label": labels}).to_csv(
    os.path.join(WORK, "test_scores.csv"), index=False)
print("submission:", sub.label.value_counts().to_dict())
for rt in WEIGHTS:
    m = test_routes == rt
    print(f"  {rt:<10} n={m.sum():>4} mean(label)={labels[m].mean():.3f}")
print("Sanity: labelled set ~55% faithful; a collapsed route means something upstream broke.")


## Notes
- Stage order matters: LoRA (transformers+bnb) runs first and is freed, so vLLM is never torn down.
- No adapter attached → the lora component is NaN and blend weights renormalize → pure v11.
- Runtime: LoRA pass ~20–40 min + 32B passes ~3–5 h → within 9 h.
- For Phase 2, attach model snapshots + wheels as datasets and run with internet OFF.
